# Gene → Anatomy Relation Pipeline

Builds a unified, deduplicated edge table for the **Gene–Anatomy** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch (Gene_Human_Anatomy_MONARCH.csv - Native Generalised)
- TARKG (Gene_Anatomy_TARKG.csv - Native Generalised)
- iBKH (Gene_Anatomy_ibkh.csv - Native Generalised)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [2]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_ANATOMY/ALL_GENE_ANATOMY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

## 1 · Mapping DBs

In [3]:
ncbi = pd.read_csv(MAPPING_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_sym2desc_dict = dict(zip(ncbi['Symbol'],  ncbi['description']))
NCBI_syn2sym_dict = {}
for synonyms, symbol in zip(ncbi['Synonyms'], ncbi['Symbol']):
    for syn in str(synonyms).split('|'):
        NCBI_syn2sym_dict[syn.strip()] = symbol
del ncbi

## 2 · Load Source Files

In [8]:
# 1. Monarch (Gene -> Anatomy)
Monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Gene_Human_Anatomy_MONARCH.csv')
Monarch.columns = Monarch.columns.str.lower()


if 'kgsource' in Monarch.columns:
    Monarch = Monarch.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'tail_name': 'tail_detail_name'})
REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

# Monarch['head_id_is'] = 'NCBI_ID'
# Monarch['tail_id_is'] = 'UBERON_ID'
Monarch['relation'] = 'Gene_AnatomicalEntity'
# Monarch['head_type'] = 'Gene'
Monarch['tail_type'] = 'AnatomicalEntity'
# Monarch['kg_source'] = 'Monarch_KG'
Monarch['kg_type'] = 'Generalised'
Monarch = Monarch[REQUIRED_COLS]
print(f"Monarch: {len(Monarch):,} rows")
Monarch


Monarch: 161,084 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,head_detail_name,tail_detail_name
0,MYBL2,Gene_AnatomicalEntity,UBERON:0003053,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,MYB proto-oncogene like 2,MYBL2,ventricular zone
1,MYBL2,Gene_AnatomicalEntity,UBERON:0000922,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,MYB proto-oncogene like 2,MYBL2,embryo
2,MYBL2,Gene_AnatomicalEntity,UBERON:0002371,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,MYB proto-oncogene like 2,MYBL2,bone marrow
3,MYBL2,Gene_AnatomicalEntity,UBERON:0001154,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,MYB proto-oncogene like 2,MYBL2,vermiform appendix
4,MYBL2,Gene_AnatomicalEntity,UBERON:0004991,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,MYB proto-oncogene like 2,MYBL2,mucosa of transverse colon
...,...,...,...,...,...,...,...,...,...,...,...,...,...
161079,RNF228,Gene_AnatomicalEntity,UBERON:0013554,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,ring finger protein 228,RNF228,Brodmann (1909) area 23
161080,RNF228,Gene_AnatomicalEntity,UBERON:0002771,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,ring finger protein 228,RNF228,middle temporal gyrus
161081,RNF228,Gene_AnatomicalEntity,UBERON:0002728,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,ring finger protein 228,RNF228,entorhinal cortex
161082,RNF228,Gene_AnatomicalEntity,UBERON:0000044,Gene,expressed_in,AnatomicalEntity,Monarch_KG,Generalised,NCBI_ID,UBERON,ring finger protein 228,RNF228,dorsal root ganglion


In [9]:
# 2. TARKG (Gene -> Anatomy)
TARKG = pd.read_csv(PROC_DIR + 'TARKG/Gene_Anatomy_TARKG.csv')
TARKG.columns = TARKG.columns.str.lower()
if 'head_name' in TARKG.columns:
    TARKG = TARKG.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in TARKG.columns:
    TARKG = TARKG.rename(columns={'tail_name': 'tail_detail_name'})

# TARKG['head_id_is'] = 'NCBI_ID'
# TARKG['tail_id_is'] = 'UBERON_ID'
TARKG['relation'] = 'Gene_AnatomicalEntity'
TARKG['head_type'] = 'Gene'
TARKG['tail_type'] = 'AnatomicalEntity'
TARKG['kg_source'] = 'TARKG'
TARKG['kg_type'] = 'Generalised'
TARKG = TARKG[REQUIRED_COLS]
print(f"TARKG: {len(TARKG):,} rows")

TARKG

TARKG: 1,683,157 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,TOP2A,Gene_AnatomicalEntity,UBERON:0002082,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,DNA topoisomerase II alpha,cardiac ventricle
1,TOP2A,Gene_AnatomicalEntity,UBERON:0002185,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,DNA topoisomerase II alpha,bronchus
2,TOP2A,Gene_AnatomicalEntity,UBERON:0002240,Gene,GENE_UNDEREXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,DNA topoisomerase II alpha,spinal cord
3,TOP2A,Gene_AnatomicalEntity,UBERON:0000955,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,DNA topoisomerase II alpha,brain
4,TOP2A,Gene_AnatomicalEntity,UBERON:0000996,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,DNA topoisomerase II alpha,vagina
...,...,...,...,...,...,...,...,...,...,...,...,...
1683152,SPANXN1,Gene_AnatomicalEntity,UBERON:0000473,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,SPANX family member N1,testis
1683153,SPANXN1,Gene_AnatomicalEntity,UBERON:0000468,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,SPANX family member N1,multicellular organism
1683154,SPANXN5,Gene_AnatomicalEntity,UBERON:0000473,Gene,GENE_EXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,SPANX family member N5,testis
1683155,SPANXN5,Gene_AnatomicalEntity,UBERON:0000473,Gene,GENE_OVEREXPRESSED_ANATOMY,AnatomicalEntity,TARKG,Generalised,NCBI_ID,UBERON_ID,SPANX family member N5,testis


In [12]:
# 3. iBKH (Gene -> Anatomy)
iBKH = pd.read_csv(PROC_DIR + 'iBKH/Gene_Anatomy_ibkh.csv')
iBKH.columns = iBKH.columns.str.lower()
if 'head_name' in iBKH.columns:
    iBKH = iBKH.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in iBKH.columns:
    iBKH = iBKH.rename(columns={'tail_name': 'tail_detail_name'})

# iBKH['head_id_is'] = 'NCBI_ID'
# iBKH['tail_id_is'] = 'UBERON_ID'
iBKH['relation'] = 'Gene_AnatomicalEntity'
iBKH['head_type'] = 'Gene'
iBKH['tail_type'] = 'AnatomicalEntity'
iBKH['kg_source'] = 'iBKH'
iBKH['relation_type'] = "NAN"
iBKH['kg_type'] = 'Generalised'
iBKH = iBKH[REQUIRED_COLS]
print(f"iBKH: {len(iBKH):,} rows")

iBKH

iBKH: 8,769,059 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,TSPAN6,Gene_AnatomicalEntity,UBERON:0000002,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,tetraspanin 6,uterine cervix
1,TSPAN6,Gene_AnatomicalEntity,UBERON:0000006,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,tetraspanin 6,islet of Langerhans
2,TSPAN6,Gene_AnatomicalEntity,UBERON:0000007,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,tetraspanin 6,pituitary gland
3,TSPAN6,Gene_AnatomicalEntity,UBERON:0000014,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,tetraspanin 6,zone of skin
4,TSPAN6,Gene_AnatomicalEntity,UBERON:0000029,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,tetraspanin 6,lymph node
...,...,...,...,...,...,...,...,...,...,...,...,...
8769054,ZRANB2-DT,Gene_AnatomicalEntity,UBERON:0000451,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,ZRANB2 divergent transcript,prefrontal cortex
8769055,ZRANB2-DT,Gene_AnatomicalEntity,UBERON:0001870,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,ZRANB2 divergent transcript,frontal cortex
8769056,ZRANB2-DT,Gene_AnatomicalEntity,UBERON:0016526,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,ZRANB2 divergent transcript,lobe of cerebral hemisphere
8769057,ZSWIM8-AS1,Gene_AnatomicalEntity,UBERON:0000341,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,ZSWIM8 antisense RNA 1,throat


## 3 · Consolidate

In [13]:
source_dfs = [Monarch, TARKG, iBKH]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']

## 4 · Gene Name Normalisation & NA Audit Before Deduplication

In [14]:
# Normalise head details using NCBI gene info mappings
mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head'] = final_df.loc[mask, 'head'].astype(str).str.replace('-', '', regex=False).map(NCBI_syn2sym_dict).fillna(final_df.loc[mask, 'head'])

mask = final_df['head_detail_name'].isna()
final_df.loc[mask, 'head_detail_name'] = final_df.loc[mask, 'head'].map(NCBI_sym2desc_dict)

final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)

print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 10,613,283

NaN audit before deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


## 5 · Deduplication & NA Audit After Deduplication

In [15]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))
group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first', 'relation_type': 'first', 'tail_type': 'first',
    'kg_source': merge_sources, 'kg_type': merge_sources,
    'head_id_is': 'first', 'tail_id_is': 'first',
    'head_detail_name': 'first', 'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
print("\nNaN audit after deduplication:")
print(final_df_dedup.isna().sum())

Before dedup: 10,613,283  |  After dedup: 8,795,493

NaN audit after deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


In [16]:
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,A1BG,Gene_AnatomicalEntity,UBERON:0000002,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,alpha-1-B glycoprotein,uterine cervix
1,A1BG,Gene_AnatomicalEntity,UBERON:0000004,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,alpha-1-B glycoprotein,nose
2,A1BG,Gene_AnatomicalEntity,UBERON:0000006,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,alpha-1-B glycoprotein,islet of Langerhans
3,A1BG,Gene_AnatomicalEntity,UBERON:0000007,Gene,expressed_in,AnatomicalEntity,Monarch_KG::iBKH,Generalised,NCBI_ID,UBERON,alpha-1-B glycoprotein,pituitary gland
4,A1BG,Gene_AnatomicalEntity,UBERON:0000010,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,alpha-1-B glycoprotein,peripheral nervous system
...,...,...,...,...,...,...,...,...,...,...,...,...
8795488,ZZZ3,Gene_AnatomicalEntity,UBERON:0035825,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,zinc finger ZZ-type containing 3,left adrenal gland cortex
8795489,ZZZ3,Gene_AnatomicalEntity,UBERON:0035827,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,zinc finger ZZ-type containing 3,right adrenal gland cortex
8795490,ZZZ3,Gene_AnatomicalEntity,UBERON:0035833,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,zinc finger ZZ-type containing 3,lower esophagus muscularis layer
8795491,ZZZ3,Gene_AnatomicalEntity,UBERON:0035834,Gene,NAN,AnatomicalEntity,iBKH,Generalised,NCBI_ID,UBERON_ID,zinc finger ZZ-type containing 3,lower esophagus mucosa


In [17]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


## 6 · Save Output

In [18]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 8,795,493 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_ANATOMY/ALL_GENE_ANATOMY.csv
